# Task 2: Velocity / Vorticity Field Reconstruction (FNO)

Self-contained training notebook for field reconstruction.
Dataset file: `../output/field_dataset.npz`


In [ ]:
from pathlib import Path
import json, numpy as np, matplotlib.pyplot as plt
import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader

# Resolve final-2 base path robustly no matter where notebook kernel starts.
CWD = Path.cwd().resolve()
if (CWD / 'final-2' / 'output').exists():
    BASE = CWD / 'final-2'
elif (CWD.name == 'notebooks') and (CWD.parent / 'output').exists():
    BASE = CWD.parent
else:
    BASE = CWD

DATA_PATH = BASE / 'output' / 'field_dataset.npz'
OUT_DIR = BASE / 'output' / 'task2_training'
OUT_DIR.mkdir(parents=True, exist_ok=True)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Base:', BASE)
print('Device:', DEVICE)


In [ ]:
ds = np.load(DATA_PATH, allow_pickle=True)
X = ds['inputs_norm'].astype(np.float32)
Y = ds['targets_norm'].astype(np.float32)
input_channels = [str(x) for x in ds['input_channels'].tolist()]
target_channels = [str(x) for x in ds['target_channels'].tolist()]
train_idx = ds['train_idx'].astype(np.int64)
val_idx = ds['val_idx'].astype(np.int64)
test_idx = ds['test_idx'].astype(np.int64)
print('X', X.shape, 'Y', Y.shape)
print('input channels:', input_channels)
print('target channels:', target_channels)


In [ ]:
# Visual sanity on one sample
zmid = X.shape[-1] // 2
plt.figure(figsize=(6,4))
plt.imshow(X[0,0,:,:,zmid], origin='lower', cmap='viridis')
plt.colorbar(); plt.title('Input sample slice'); plt.tight_layout(); plt.show()

plt.figure(figsize=(6,4))
plt.imshow(Y[0,0,:,:,zmid], origin='lower', cmap='turbo')
plt.colorbar(); plt.title('Target sample slice'); plt.tight_layout(); plt.show()


In [ ]:
class GridDataset(Dataset):
    def __init__(self, X, Y, idx):
        self.X = X[idx]
        self.Y = Y[idx]
    def __len__(self):
        return self.X.shape[0]
    def __getitem__(self, i):
        return torch.from_numpy(self.X[i]), torch.from_numpy(self.Y[i])

train_loader = DataLoader(GridDataset(X,Y,train_idx), batch_size=2, shuffle=True)
val_loader = DataLoader(GridDataset(X,Y,val_idx), batch_size=2, shuffle=False)
test_loader = DataLoader(GridDataset(X,Y,test_idx), batch_size=2, shuffle=False)


In [ ]:
try:
    from neuralop.models import FNO
except Exception as exc:
    raise RuntimeError('Need neuraloperator package for FNO.') from exc

def rel_l2(pred, tgt, eps=1e-12):
    d = (pred - tgt).reshape(pred.shape[0], -1)
    t = tgt.reshape(tgt.shape[0], -1)
    return (torch.linalg.norm(d, dim=1) / torch.linalg.norm(t, dim=1).clamp_min(eps)).mean()

model = FNO(
    n_modes=(12,12,12),
    in_channels=X.shape[1],
    out_channels=Y.shape[1],
    hidden_channels=32,
    n_layers=4,
    non_linearity=torch.nn.functional.relu,
).to(DEVICE)
print('params:', sum(p.numel() for p in model.parameters() if p.requires_grad))


In [ ]:
opt_cls = None
try:
    import neuralop.training as nt
    opt_cls = getattr(nt, 'AdamW', None) or getattr(nt, 'Adam', None)
except Exception:
    pass
if opt_cls is None:
    opt_cls = torch.optim.AdamW

opt = opt_cls(model.parameters(), lr=1e-3, weight_decay=1e-6)
sch = torch.optim.lr_scheduler.StepLR(opt, step_size=100, gamma=0.5)


In [ ]:
EPOCHS = 300
hist = []
best = np.inf
best_state = None

def eval_loader(loader):
    model.eval()
    rels, mses = [], []
    with torch.no_grad():
        for x,y in loader:
            x, y = x.to(DEVICE), y.to(DEVICE)
            p = model(x)
            rels.append(float(rel_l2(p,y).item()))
            mses.append(float(torch.mean((p-y)**2).item()))
    return float(np.mean(rels)), float(np.mean(mses))

for ep in range(1, EPOCHS+1):
    model.train()
    losses = []
    for x,y in train_loader:
        x, y = x.to(DEVICE), y.to(DEVICE)
        opt.zero_grad(set_to_none=True)
        p = model(x)
        loss = rel_l2(p,y)
        loss.backward()
        opt.step()
        losses.append(float(loss.item()))
    sch.step()

    tr = float(np.mean(losses))
    vr, vm = eval_loader(val_loader)
    hist.append({'epoch':ep, 'train_loss':tr, 'val_rel_l2':vr, 'val_mse':vm})

    if vr < best:
        best = vr
        best_state = {k:v.detach().cpu() for k,v in model.state_dict().items()}

    if ep == 1 or ep % 10 == 0:
        print(f'[{ep:03d}] train={tr:.6f} val_rel={vr:.6f} val_mse={vm:.6f}')

if best_state is not None:
    model.load_state_dict(best_state)
tr, tm = eval_loader(test_loader)
print('TEST rel_l2=', tr, 'mse=', tm)


In [ ]:
torch.save(model.state_dict(), OUT_DIR / 'best_field_fno_model.pt')
(OUT_DIR / 'history.json').write_text(json.dumps(hist, indent=2))

plt.figure(figsize=(7,4))
plt.plot([h['epoch'] for h in hist], [h['train_loss'] for h in hist], label='train')
plt.plot([h['epoch'] for h in hist], [h['val_rel_l2'] for h in hist], label='val_rel_l2')
plt.xlabel('epoch'); plt.ylabel('loss'); plt.grid(alpha=0.3); plt.legend(); plt.tight_layout(); plt.show()

# quick pred vs true on first test sample
i = int(test_idx[0])
with torch.no_grad():
    p = model(torch.from_numpy(X[i:i+1]).to(DEVICE)).cpu().numpy()[0]
y = Y[i]
z = y.shape[-1] // 2

plt.figure(figsize=(14,4))
plt.subplot(1,3,1); plt.imshow(y[0,:,:,z], origin='lower', cmap='turbo'); plt.title('true'); plt.colorbar()
plt.subplot(1,3,2); plt.imshow(p[0,:,:,z], origin='lower', cmap='turbo'); plt.title('pred'); plt.colorbar()
plt.subplot(1,3,3); plt.imshow(np.abs(p[0,:,:,z]-y[0,:,:,z]), origin='lower', cmap='magma'); plt.title('abs err'); plt.colorbar()
plt.tight_layout(); plt.show()

print('saved to', OUT_DIR)
